> [!TIP]
> **Active Development Copy (v2):** This notebook is the workspace for ongoing improvements and extensions to **Enterprise Synthetic Data Generation & Scenario Injection**.
> The verified baseline remains preserved in `v1_baseline.ipynb` and `../../01_generate_enterprise_data.ipynb`.

# Enterprise Synthetic Data Generation & Scenario Injection

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 01 of 12  
**Focus:** Synthetic Enterprise Architecture, Relational Topologies, Controlled Scenario Injection, Referential Auditing  

---

## 1. Business Problem

In enterprise IT Financial Management (ITFM) and Technology Business Management (TBM), analytical models are often starved of representative data. Real-world corporate financial ledgers, Configuration Management Databases (CMDBs), and vendor contracts contain proprietary data, trade secrets, and compliance-restricted identifiers. Conversely, standard synthetic test datasets typically suffer from two severe shortcomings:
1. **Random Noise Without Topology:** Randomly generating applications, costs, and servers fails to capture the intricate hierarchical relationships of enterprise architectures (e.g., applications supporting capabilities, running on specific middleware, procured under multi-year vendor contracts).
2. **Absence of Analytic Scenarios:** Pure random generation lacks the empirical anomalies, bottlenecks, and variances that analytical platforms are built to diagnose—such as underutilized legacy applications, capability overlap, cost anomalies, or failed transformation projects.

To build and validate a portfolio-quality **Technology Value Intelligence (TVI)** engine, we must generate a complete, deterministic, referentially intact enterprise landscape embedding controlled business scenarios.

---

## 2. Why Dummy / Synthetic Data is Being Used

Synthetic data provides three decisive advantages for this research prototype:
- **Zero Privacy & Legal Risk:** Fictional organization names, synthetic financials, and artificial server topologies ensure zero data exposure risk.
- **Reproducibility & Auditability:** By pinning mathematical pseudorandom number generators to fixed seeds, researchers and auditors can replicate every calculation down to the exact rupee and transaction count.
- **Controlled Stress Testing:** We can programmatically inject known failure modes (e.g. single points of failure, budget overruns, unbacked rate spikes) to verify whether downstream graph and financial algorithms correctly detect them.

---

## 3. Enterprise Assumptions & Scale Definition

We model a mid-to-large diversified financial services institution operating across retail, commercial, wealth management, and payment sectors:
- **Currency:** Indian Rupee (INR, ₹), reported in standard Crores ($1\text{ Cr} = 10,000,000$) and Lakhs ($1\text{ L} = 100,000$).
- **Reporting Period:** 12 monthly periods representing FY2024 (2024-01 through 2024-12).
- **Enterprise Scale:**
  - $10$ Business Units (e.g., Retail Banking, Wealth Management, Digital Payments).
  - $25$ Business Capabilities (e.g., Order-to-Cash, Procure-to-Pay, Fraud Detection).
  - $18$ IT Services delivering standardized technology offerings.
  - $50$ Applications spanning core banking, CRM, ERP, and payment microservices.
  - $35$ Infrastructure and runtime Technologies (Cloud, Databases, Middleware, OS).
  - $10$ Third-party strategic Vendors with multi-year contract values.
  - $40$ Discretionary Projects / Investments (Modernization, Cloud Migration, AI).
  - $12$ Months of granular General Ledger Cost Records and Operational Consumption Logs.

---

## 4. Deterministic Random Seed Specification

To guarantee mathematical determinism across all operating systems and Python executions, we establish a global seed:
$$\text{RANDOM\_SEED} = 42$$
This ensures that all NumPy distributions, random strides, and scenario injections remain identical on every execution.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Ensure TVI package is importable
# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config, get_project_root
from tvi.data_generation import EnterpriseDataGenerator
from tvi.validation import DataQualityValidator

config = load_config()
print(f"Project Name : {config.project.name} (v{config.project.version})")
print(f"Random Seed  : {config.project.random_seed}")
print(f"Currency     : {config.project.currency} ({config.project.currency_symbol})")
print(f"Target Year  : {config.project.reporting_year}")

Project Name : Technology Value Intelligence (v1.0.0)
Random Seed  : 42
Currency     : INR (₹)
Target Year  : 2024


---

## 5. Entity Generation Walkthrough

The generation process executes progressively across foundational entities:
1. `BusinessUnit`: Organizational divisions consuming technology and generating commercial revenue.
2. `BusinessCapability`: Functional domains executed by the enterprise.
3. `ITService`: Standardized technology service offerings cataloged and delivered by IT.
4. `Vendor`: Fictional technology and services providers.
5. `Technology`: Infrastructure, runtime, database, and cloud components.
6. `Application`: Core software systems deployed to deliver services.
7. `Project`: Strategic discretionary investments and capital transformation programs.

In [2]:
# Instantiate the deterministic generator
generator = EnterpriseDataGenerator(config)

# 1. Master Entities
bu_df = generator.generate_business_units()
cap_df = generator.generate_business_capabilities(bu_df)
srv_df = generator.generate_it_services()
ven_df = generator.generate_vendors()
tech_df = generator.generate_technologies(ven_df)
app_df = generator.generate_applications(srv_df, ven_df)
prj_df = generator.generate_projects(bu_df)

print(f"Generated Entities:")
print(f"  • Business Units        : {len(bu_df):3d} rows")
print(f"  • Business Capabilities : {len(cap_df):3d} rows")
print(f"  • IT Services           : {len(srv_df):3d} rows")
print(f"  • Vendors               : {len(ven_df):3d} rows")
print(f"  • Technologies          : {len(tech_df):3d} rows")
print(f"  • Applications          : {len(app_df):3d} rows")
print(f"  • Projects              : {len(prj_df):3d} rows")

Generated Entities:
  • Business Units        :  10 rows
  • Business Capabilities :  25 rows
  • IT Services           :  18 rows
  • Vendors               :  10 rows
  • Technologies          :  35 rows
  • Applications          :  50 rows
  • Projects              :  40 rows


---

## 6. Relationship Generation & Topological Mappings

Next, we generate the multi-relational edges that bind these entities into an interconnected enterprise graph:
- `rel_app_technology`: Applications running on underlying infrastructure.
- `rel_app_capability`: Applications supporting business capabilities with explicit support types (`Primary`, `Secondary`, `Overlapping`).
- `rel_app_dependency`: App-to-app dependencies embedding Scenario D (central bottleneck).
- `rel_project_capability`: Projects targeting capability transformations.
- `rel_app_project`: Applications modernized or funded by projects.

In [3]:
rel_app_tech = generator.generate_app_technology_relations(app_df, tech_df)
rel_app_cap = generator.generate_app_capability_relations(app_df, cap_df)
rel_app_dep = generator.generate_app_dependencies(app_df)
rel_prj_cap = generator.generate_project_capability_relations(prj_df, cap_df)
rel_app_prj = generator.generate_app_project_relations(app_df, prj_df)

print("Generated Relational Mappings:")
print(f"  • App -> Technology Mappings   : {len(rel_app_tech):4d} edges")
print(f"  • App -> Capability Mappings   : {len(rel_app_cap):4d} edges")
print(f"  • App -> App Dependencies      : {len(rel_app_dep):4d} edges")
print(f"  • Project -> Capability Links  : {len(rel_prj_cap):4d} edges")
print(f"  • Project -> App Funding Links : {len(rel_app_prj):4d} edges")

Generated Relational Mappings:
  • App -> Technology Mappings   :  151 edges
  • App -> Capability Mappings   :   72 edges
  • App -> App Dependencies      :   32 edges
  • Project -> Capability Links  :   60 edges
  • Project -> App Funding Links :   40 edges


---

## 7. Monthly General Ledger Cost Data Generation

Monthly costs are generated across standard TBM cost pools (`Software`, `Internal Labor`, `Outside Services`) and categories (`People`, `Software`, `Hardware`, `Cloud`, `Infrastructure`).
We explicitly incorporate:
- Dedicated direct costs attributed to applications.
- Shared service costs attributed to IT services where `application_id IS NULL`.
- Scenario G: Spend surges in August for `APP014` due to transaction scaling.
- Scenario H: Spend jumps in August for `APP022` with flat utilization.

In [4]:
cost_df = generator.generate_cost_records(
    app_df, srv_df, bu_df, ven_df, prj_df, rel_app_cap
)
print(f"Generated Cost Records: {len(cost_df):5d} ledger rows")
print(f"Total Annual Technology Spend: ₹{cost_df['amount'].sum()/1e7:.2f} Crores")
display(cost_df.head(4))

Generated Cost Records:  2016 ledger rows
Total Annual Technology Spend: ₹161.83 Crores


---

## 8. Monthly Operational Consumption Telemetry Generation

Operational telemetry tracks real-world usage across 6 dimensions:
$$\mathbf{C} = [\text{Active Users}, \text{Transactions}, \text{API Calls}, \text{Compute Hours}, \text{Storage GB}, \text{Service Tickets}]$$
Correlations are mathematically enforced:
- Higher transactions scale compute hours and API volume.
- Deliberate anomalies are injected (Scenario B with low users/high cost, Scenario G with scaling transactions, Scenario H with flat usage).

In [5]:
cons_df = generator.generate_consumption_records(app_df, bu_df)
print(f"Generated Consumption Records: {len(cons_df):5d} telemetry rows")
display(cons_df.head(4))

Generated Consumption Records:   792 telemetry rows


---

## 9. Discretionary Projects Generation
Projects represent capital and operational transformation investments across 8 project types: Modernization, Cloud Migration, AI, Application Replacement, Automation, Regulatory, and Infrastructure.

In [6]:
print("Sample Projects:")
display(prj_df[["project_id", "project_name", "project_type", "sponsor_business_unit", "investment_budget", "actual_spend", "status"]].head(5))

Sample Projects:


---

## 10. Expected and Realized Benefits Generation
Benefits model the financial and operational outcomes delivered by completed and in-progress initiatives:
- `Cost Reduction`, `Productivity`, `Revenue Enablement`, `Risk Reduction`, `Cycle Time Reduction`, `Capacity Release`, `Customer Experience`.
- Embeds Scenario E (PRJ003 exceeded benefit realization) and Scenario F (PRJ014 severe benefit gap).

In [7]:
ben_df = generator.generate_benefits(prj_df)
print(f"Generated Benefits Records: {len(ben_df):4d} entries")
display(ben_df.head(4))

Generated Benefits Records:   80 entries


---

## 11. KPI Catalog & Progression Generation
Key Performance Indicators track quantitative performance shifts (e.g., Cost per Transaction, Incident Resolution Time, Core Banking Availability).

In [8]:
kpi_df = generator.generate_kpis()
rel_ben_kpi = generator.generate_benefit_kpi_relations(ben_df, kpi_df)

print(f"Generated KPIs: {len(kpi_df):3d} indicators | Benefit-to-KPI Links: {len(rel_ben_kpi):4d}")
display(kpi_df.head(5))

Generated KPIs:  15 indicators | Benefit-to-KPI Links:   80


---

## 12. Referential Integrity & Validation Audit

Before persisting data to disk, we execute a rigorous validation suite verifying:
1. **Uniqueness:** All primary keys are unique.
2. **Null Checks:** Mandatory foreign and primary keys contain zero null values.
3. **Numeric Sanity:** Non-negative costs, users, transactions, and storage.
4. **Temporal Sanity:** Project end dates must be on or after start dates.
5. **Referential Integrity:** All foreign keys link to valid parent entity IDs.

In [9]:
all_datasets = {
    "business_units": bu_df,
    "business_capabilities": cap_df,
    "it_services": srv_df,
    "vendors": ven_df,
    "technologies": tech_df,
    "applications": app_df,
    "projects": prj_df,
    "rel_app_technology": rel_app_tech,
    "rel_app_capability": rel_app_cap,
    "rel_app_dependency": rel_app_dep,
    "rel_project_capability": rel_prj_cap,
    "rel_app_project": rel_app_prj,
    "benefits": ben_df,
    "kpis": kpi_df,
    "rel_benefit_kpi": rel_ben_kpi,
    "cost_records": cost_df,
    "consumption_records": cons_df,
}

validator = DataQualityValidator(all_datasets)
report = validator.validate_all()

print(f"=== Data Quality Audit Report ===")
print(f"Status        : {report['status']}")
print(f"Checks Run    : {report['total_checks']}")
print(f"Issues Logged : {len(report['issues'])}")
if report["issues"]:
    for iss in report["issues"]:
        print(f"  [X] {iss}")
else:
    print("✓ Zero data-quality defects. All integrity constraints verified.")

=== Data Quality Audit Report ===
Status        : PASS
Checks Run    : 5
Issues Logged : 0
✓ Zero data-quality defects. All integrity constraints verified.


---

## 13. Exporting Datasets to CSV

We export all 17 DataFrames into `data/generated/` as CSV files, serving as the raw landing zone for DuckDB and NetworkX.

In [10]:
out_dir = get_project_root() / "data" / "generated"
out_dir.mkdir(parents=True, exist_ok=True)

for name, df in all_datasets.items():
    csv_file = out_dir / f"{name}.csv"
    df.to_csv(csv_file, index=False)
    print(f"Exported: {csv_file.name:28s} ({len(df):5d} rows)")

Exported: business_units.csv           (   10 rows)
Exported: business_capabilities.csv    (   25 rows)
Exported: it_services.csv              (   18 rows)
Exported: vendors.csv                  (   10 rows)
Exported: technologies.csv             (   35 rows)
Exported: applications.csv             (   50 rows)
Exported: projects.csv                 (   40 rows)
Exported: rel_app_technology.csv       (  151 rows)
Exported: rel_app_capability.csv       (   72 rows)
Exported: rel_app_dependency.csv       (   32 rows)
Exported: rel_project_capability.csv   (   60 rows)
Exported: rel_app_project.csv          (   40 rows)
Exported: benefits.csv                 (   80 rows)
Exported: kpis.csv                     (   15 rows)
Exported: rel_benefit_kpi.csv          (   80 rows)
Exported: cost_records.csv             ( 2016 rows)
Exported: consumption_records.csv      (  792 rows)


---

## 14. Verification of All 8 Analytical Scenarios (A through H)

We perform formal empirical assertions validating that our synthetic dataset embeds all 8 target scenarios:

In [11]:
# Assertions and Proofs for Scenarios A through H
scenario_results = []

# Scenario A: APP001 Scale Anchor
app001_cost = cost_df[cost_df["application_id"] == "APP001"]["amount"].sum()
app001_users = cons_df[cons_df["application_id"] == "APP001"]["active_users"].mean()
scenario_results.append({
    "Scenario": "Scenario A (Scale Anchor)",
    "Entity": "APP001 (CoreBanking Alpha)",
    "Evidence": f"Annual Cost: ₹{app001_cost/1e7:.2f} Cr, Avg Users: {app001_users:.0f}",
    "Verification": "PASS" if app001_cost > 1e8 and app001_users > 8000 else "FAIL"
})

# Scenario B: APP021 Underutilized Legacy
app021_cost = cost_df[cost_df["application_id"] == "APP021"]["amount"].sum()
app021_users = cons_df[cons_df["application_id"] == "APP021"]["active_users"].mean()
scenario_results.append({
    "Scenario": "Scenario B (Underutilized Sprawl)",
    "Entity": "APP021 (Legacy Web Portal)",
    "Evidence": f"Annual Cost: ₹{app021_cost/1e7:.2f} Cr, Avg Users: {app021_users:.0f} (Cost/User: ₹{app021_cost/app021_users:,.0f})",
    "Verification": "PASS" if app021_cost > 5e7 and app021_users < 200 else "FAIL"
})

# Scenario C: CAP003 Duplicate Capability Overlap
otc_apps = rel_app_cap[rel_app_cap["capability_id"] == "CAP003"]["application_id"].tolist()
scenario_results.append({
    "Scenario": "Scenario C (Capability Duplication)",
    "Entity": "CAP003 (Order-to-Cash)",
    "Evidence": f"Supported by multiple apps: {', '.join(otc_apps)}",
    "Verification": "PASS" if len(otc_apps) >= 2 else "FAIL"
})

# Scenario D: APP010 Single Point of Failure Bottleneck
app010_in_deps = rel_app_dep[rel_app_dep["target_app_id"] == "APP010"]["source_app_id"].tolist()
scenario_results.append({
    "Scenario": "Scenario D (SPOF Hub)",
    "Entity": "APP010 (Central Auth Hub)",
    "Evidence": f"Inbound dependent applications: {len(app010_in_deps)} apps ({', '.join(app010_in_deps[:4])}...)",
    "Verification": "PASS" if len(app010_in_deps) >= 6 else "FAIL"
})

# Scenario E: PRJ003 High Realization Investment
prj003_exp = prj_df[prj_df["project_id"] == "PRJ003"]["expected_annual_benefit"].values[0]
prj003_rel = ben_df[ben_df["project_id"] == "PRJ003"]["realized_value"].sum()
scenario_results.append({
    "Scenario": "Scenario E (High TVR Investment)",
    "Entity": "PRJ003 (Payment Modernization)",
    "Evidence": f"Expected: ₹{prj003_exp/1e7:.1f} Cr | Realized: ₹{prj003_rel/1e7:.1f} Cr ({prj003_rel/prj003_exp*100:.1f}%)",
    "Verification": "PASS" if prj003_rel >= prj003_exp else "FAIL"
})

# Scenario F: PRJ014 Benefit Realization Gap
prj014_exp = prj_df[prj_df["project_id"] == "PRJ014"]["expected_annual_benefit"].values[0]
prj014_rel = ben_df[ben_df["project_id"] == "PRJ014"]["realized_value"].sum()
scenario_results.append({
    "Scenario": "Scenario F (Benefit Gap Overrun)",
    "Entity": "PRJ014 (Legacy CRM Consolidation)",
    "Evidence": f"Expected: ₹{prj014_exp/1e7:.1f} Cr | Realized: ₹{prj014_rel/1e7:.1f} Cr ({prj014_rel/prj014_exp*100:.1f}%)",
    "Verification": "PASS" if (prj014_rel / prj014_exp) < 0.35 else "FAIL"
})

# Scenario G: APP014 Elastic Scaling
app014_cost_jul = cost_df[(cost_df["application_id"] == "APP014") & (cost_df["month"] == "2024-07")]["amount"].sum()
app014_cost_aug = cost_df[(cost_df["application_id"] == "APP014") & (cost_df["month"] == "2024-08")]["amount"].sum()
app014_txns_jul = cons_df[(cons_df["application_id"] == "APP014") & (cons_df["month"] == "2024-07")]["transactions"].sum()
app014_txns_aug = cons_df[(cons_df["application_id"] == "APP014") & (cons_df["month"] == "2024-08")]["transactions"].sum()
scenario_results.append({
    "Scenario": "Scenario G (Elastic Scale Surge)",
    "Entity": "APP014 (Cloud Payment Hub)",
    "Evidence": f"Cost shift: ₹{app014_cost_jul/1e5:.1f}L -> ₹{app014_cost_aug/1e5:.1f}L (+{(app014_cost_aug-app014_cost_jul)/app014_cost_jul*100:.0f}%) | Txns: +{(app014_txns_aug-app014_txns_jul)/app014_txns_jul*100:.0f}%",
    "Verification": "PASS" if app014_cost_aug > app014_cost_jul and app014_txns_aug > app014_txns_jul else "FAIL"
})

# Scenario H: APP022 Rate/Infra Spend Anomaly
app022_cost_jul = cost_df[(cost_df["application_id"] == "APP022") & (cost_df["month"] == "2024-07")]["amount"].sum()
app022_cost_aug = cost_df[(cost_df["application_id"] == "APP022") & (cost_df["month"] == "2024-08")]["amount"].sum()
app022_txns_jul = cons_df[(cons_df["application_id"] == "APP022") & (cons_df["month"] == "2024-07")]["transactions"].sum()
app022_txns_aug = cons_df[(cons_df["application_id"] == "APP022") & (cons_df["month"] == "2024-08")]["transactions"].sum()
scenario_results.append({
    "Scenario": "Scenario H (Unbacked Anomaly)",
    "Entity": "APP022 (Batch Billing Engine)",
    "Evidence": f"Cost shift: ₹{app022_cost_jul/1e5:.1f}L -> ₹{app022_cost_aug/1e5:.1f}L (+{(app022_cost_aug-app022_cost_jul)/app022_cost_jul*100:.0f}%) | Txns: flat ({(app022_txns_aug-app022_txns_jul)/app022_txns_jul*100:.1f}%)",
    "Verification": "PASS" if app022_cost_aug > (app022_cost_jul * 1.5) and abs(app022_txns_aug - app022_txns_jul) / app022_txns_jul < 0.1 else "FAIL"
})

scenarios_df = pd.DataFrame(scenario_results)
display(scenarios_df)
assert (scenarios_df["Verification"] == "PASS").all(), "All scenarios must verify PASS!"

---

## 15. Statistical Telemetry Distributions

We evaluate the statistical distributions of our telemetry variables (Users, Transactions, API Calls, Compute Hours) across the application portfolio:

In [12]:
# Telemetry Distribution Histograms
app_telemetry = cons_df.groupby("application_id").agg({
    "active_users": "mean",
    "transactions": "sum",
    "api_calls": "sum",
    "compute_hours": "mean"
}).reset_index()

fig, axs = plt.subplots(2, 2, figsize=(11, 7))

axs[0, 0].hist(app_telemetry["active_users"], bins=15, color="#1f77b4", edgecolor="black", alpha=0.8)
axs[0, 0].set_title("Distribution of Average Active Users", fontsize=11)
axs[0, 0].set_xlabel("Active Users")
axs[0, 0].grid(True, linestyle="--", alpha=0.4)

axs[0, 1].hist(app_telemetry["transactions"] / 1e6, bins=15, color="#2ca02c", edgecolor="black", alpha=0.8)
axs[0, 1].set_title("Distribution of Annual Transactions (Millions)", fontsize=11)
axs[0, 1].set_xlabel("Transactions (M)")
axs[0, 1].grid(True, linestyle="--", alpha=0.4)

axs[1, 0].hist(app_telemetry["api_calls"] / 1e6, bins=15, color="#ff7f0e", edgecolor="black", alpha=0.8)
axs[1, 0].set_title("Distribution of Annual API Calls (Millions)", fontsize=11)
axs[1, 0].set_xlabel("API Calls (M)")
axs[1, 0].grid(True, linestyle="--", alpha=0.4)

axs[1, 1].hist(app_telemetry["compute_hours"], bins=15, color="#d62728", edgecolor="black", alpha=0.8)
axs[1, 1].set_title("Distribution of Monthly Compute Hours", fontsize=11)
axs[1, 1].set_xlabel("Compute Hours")
axs[1, 1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

---

## 16. Active Development (v2): Advanced Enterprise Data Synthesis

In the **v2 Architecture**, we expand enterprise data synthesis beyond foundational tabular entities:
1. **Declarative YAML Scenario Configuration:** Parameterized scenario bounds loaded from `config/scenarios.yaml`.
2. **Granular Cloud Billing Records (AWS CUR / Azure Cost Management):** Multi-cloud line-item infrastructure invoices linking resource IDs to application tags.
3. **Multi-Level Organizational Hierarchy:** 3-tier organizational structure linking Enterprise Divisions, Business Units, and Cost Centers.
4. **Organic Data Drift & Seasonal Cycles:** Simulating multi-period compound drift, Q4 fiscal ramp, and stochastic variance.

In [13]:
# 1. Inspect Declarative Scenario Configuration from YAML
from tvi.data_generation import EnterpriseDataGenerator
import pandas as pd

generator = EnterpriseDataGenerator()
print(f"Loaded {len(generator.scenarios)} declarative business scenarios:")
for sc_id, sc_data in generator.scenarios.items():
    print(f"  • {sc_id.upper()}: {sc_data.get('name', '')} -> {sc_data.get('description', '')[:70]}...")

# 2. Generate Granular Cloud Billing Records (AWS CUR / Azure)
cur_df = generator.generate_cloud_billing_records(app_df)
print(f"\nGenerated {len(cur_df):,} Cloud Billing Line-Items (AWS CUR / Azure Cost Management)")
display(cur_df[["line_item_id", "month", "cloud_provider", "line_item_product_code", "pricing_unit", "usage_amount", "line_item_unblended_cost", "application_id"]].head(5))

# 3. Generate 3-Tier Organizational Hierarchy & Cost Centers
cost_centers_df = generator.generate_organizational_hierarchy(bu_df)
print(f"\nGenerated {len(cost_centers_df)} Cost Centers across {bu_df['business_unit_id'].nunique()} Business Units")
display(cost_centers_df.head(6))

# 4. Simulate Multi-Period Data Drift & Seasonal Progression
drift_cost_df, drift_cons_df = generator.apply_data_drift(cost_df, cons_df, monthly_trend=0.02, seasonality=True)
drift_monthly = drift_cost_df.groupby("month")["amount"].sum() / 1e7
orig_monthly = cost_df.groupby("month")["amount"].sum() / 1e7

drift_comparison = pd.DataFrame({
    "Baseline Monthly Spend (₹ Cr)": orig_monthly,
    "Drifted & Seasonal Spend (₹ Cr)": drift_monthly,
    "Variance (₹ Cr)": (drift_monthly - orig_monthly).round(3)
})
display(drift_comparison)

Loaded 8 declarative business scenarios:
  • SCENARIO_A: Enterprise Core Banking Engine -> Scale Anchor: High absolute TCO with high operational utilization and ...
  • SCENARIO_B: Legacy Branch Portal -> Underutilized Legacy: Expensive to operate with low active users and h...
  • SCENARIO_C: Order-to-Cash -> Duplicate Capability Overlap: Redundant functional support across mult...
  • SCENARIO_D: Central Authentication Hub -> Single Point of Failure (SPOF): High inbound reverse-dependency blast ...
  • SCENARIO_E: Real-Time Payment Modernization -> Successful Investment: Completed under budget with over-achieved benef...
  • SCENARIO_F: Global Legacy CRM Consolidation -> Benefit Gap Overrun: Completed over budget with severe benefit shortfa...
  • SCENARIO_G: Digital Payment Gateway -> Elastic Consumption Surge: Month 8 spend surge proportional to transac...
  • SCENARIO_H: Wealth Advisory Engine -> Unbacked Cost Anomaly: Month 8 spend surge with flat transaction volum...

Generated 

## 16. Limitations & Methodological Guardrails

1. **Synthetic Determinism:** While the dataset realistically models enterprise topologies, it is synthetic. It should not be used as an empirical benchmark for production sizing without domain adaptation.
2. **Discrete Monthly Aggregations:** Telemetry is sampled in monthly intervals. Real-time intraday autoscaling spikes are smoothed into monthly compute totals.

## 17. Next Step

In **Notebook 02: TBM / ITFM Data Model**, we ingest these CSV datasets into **DuckDB** to build a high-performance in-process columnar warehouse, establishing formal cost pools, IT towers, and financial allocation models.